# LELA60342 Week 5 - Evaluation: Bootstrapping and Cross-validation

### SEATS: 335771

# Bootstrapping a single classifier

In [ ]:
import numpy as np
## Create simulated data
#np.random.seed(10)
w1_center = (2, 3)
w2_center = (3, 2)
batch_size=1000

x = np.zeros((batch_size, 2))
y = np.zeros(batch_size)
for i in range(batch_size):
    if np.random.random() > 0.5:
        x[i] = np.random.normal(loc=w1_center,scale=10)
    else:
        x[i] = np.random.normal(loc=w2_center,scale=10)
        y[i] = 1




In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=10)
logisticRegr = LogisticRegression()
logisticRegr.fit(x_train, y_train)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
precision_recall_fscore_support(y_test,logisticRegr.predict(x_test),average="macro")[2]

In [ ]:
import random
import numpy as np
import math

def draw_bootstrap_sample(data):
      n=len(data[0])
      indices = random.choices(range(n),k=n)
      return((data[0][indices],data[1][indices]))

def bootstrap_fscore(data, model, num_samples):
    scores = []
    for i in range(num_samples):
        data_bs = draw_bootstrap_sample((data[0],data[1]))
        score=precision_recall_fscore_support(data_bs[1],model.predict(data_bs[0]),average="macro")[2]
        scores.append(score)
    scores=np.sort(scores)
    return (np.mean(scores),scores[math.floor(len(scores)*0.025)],scores[math.floor(len(scores)*0.975)])

In [ ]:
bootstrap_fscore((x_test,y_test),logisticRegr,1000)

# K-fold cross-validation

In [ ]:
from sklearn.model_selection import KFold

In [ ]:
kf=KFold(5)
for i, (train_index, test_index) in enumerate(kf.split(x)):
    print(f"Fold {i}:")
    print(f"  Training dataset index: {train_index}")
    print(f"  Test dataset index: {test_index}")

In [ ]:
kf=KFold(5)

logisticRegr = LogisticRegression()
scores=[]
for i, (train_index, test_index) in enumerate(kf.split(x)):
  logisticRegr = LogisticRegression()
  x_train = x[train_index]
  y_train = y[train_index]
  x_test = x[test_index]
  y_test = y[test_index]
  logisticRegr.fit(x_train, y_train)
  scores.append(precision_recall_fscore_support(y_test,logisticRegr.predict(x_test),average="macro")[2])

scores=np.sort(scores)
print(np.mean(scores),scores[math.floor(len(scores)*0.025)],scores[math.floor(len(scores)*0.975)])

In [ ]:
kf=KFold(100)

logisticRegr = LogisticRegression()
scores=[]
for i, (train_index, test_index) in enumerate(kf.split(x)):
  logisticRegr = LogisticRegression()
  x_train = x[train_index]
  y_train = y[train_index]
  x_test = x[test_index]
  y_test = y[test_index]
  logisticRegr.fit(x_train, y_train)
  scores.append(precision_recall_fscore_support(y_test,logisticRegr.predict(x_test),average="macro")[2])

scores=np.sort(scores)
print(np.mean(scores),scores[math.floor(len(scores)*0.025)],scores[math.floor(len(scores)*0.975)])

# Comparing Classifiers

In [ ]:
import numpy as np
## Create simulated data
np.random.seed(4)
w1_center = (2, 3, 1, 9, 11)
w2_center = (3, 2, 8, 5, 7)
batch_size=1000

x = np.zeros((batch_size, 5))
y = np.zeros(batch_size)
for i in range(batch_size):
    if np.random.random() > 0.5:
        x[i] = np.random.normal(loc=w1_center,scale=5)
    else:
        x[i] = np.random.normal(loc=w2_center,scale=5)
        y[i] = 1


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=10)


In [ ]:
logisticRegr = LogisticRegression()
logisticRegr.fit(x_train, y_train)

In [ ]:
precision_recall_fscore_support(y_test,logisticRegr.predict(x_test),average="macro")[2]

In [ ]:
MLP_2 = MLPClassifier(3,max_iter=1000)
MLP_2.fit(x_train, y_train)

In [ ]:
precision_recall_fscore_support(y_test,MLP_2.predict(x_test),average="macro")[2]

In [ ]:
import random
import numpy as np

def draw_bootstrap_sample(data):
      n=len(data[0])
      indices = random.choices(range(n),k=n)
      return((data[0][indices],data[1][indices]))

def bootstrap_fscore_diff(data, model1,model2, num_samples):
    scores = []
    for i in range(num_samples):
        data_bs = draw_bootstrap_sample((data[0],data[1]))
        score1=precision_recall_fscore_support(data_bs[1],model1.predict(data_bs[0]),average="macro")[2]
        score2=precision_recall_fscore_support(data_bs[1],model2.predict(data_bs[0]),average="macro")[2]
        scores.append(score1-score2)
    scores=np.sort(scores)
    return (np.mean(scores),scores[round(len(scores)*0.025)],scores[round(len(scores)*0.975)])

In [ ]:
bootstrap_fscore_diff((x_test,y_test),logisticRegr,MLP_2,1000)

# Using R within Python

I am including this as it might be something you want to do at some point. The reverse is also possible: you can call Python from within R using the R package *reticulate* (https://rstudio.github.io/reticulate/)

In [ ]:
!pip install rpy2

In [ ]:
from rpy2.robjects import r
import rpy2.robjects as robjects
import pandas as pd
from rpy2.robjects import pandas2ri

In [ ]:
df=pd.Series([1,2,3,4])
with (robjects.default_converter + pandas2ri.converter).context():
  r_from_pd_df = robjects.conversion.get_conversion().py2rpy(df)

In [ ]:
r('''
samplemean <- function(x, d) {
    return(mean(x[d]))
}
''')

In [ ]:
sm = robjects.globalenv['samplemean']
print(sm.r_repr())

In [ ]:
r('''
library(boot)
''')

In [ ]:
r.boot(r_from_pd_df,sm,100)

# Evaluating a sentiment classifier

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/coursework/Compiled_Reviews.txt

In [ ]:
import pandas as pd
import torch
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
df=pd.read_table("Compiled_Reviews.txt")
df=df.sample(n=10000, random_state=1)
reviews=[]
review_lengths=[]
for i,row in df.iterrows():
    review=str(row.REVIEW).split()
    review=[x.lower() for x in review]
    review = [word for word in review if word not in stop_words]
    review_lengths.append(len(review))
    reviews.extend(review)
vocab_list = list(set(reviews))
vocab={}
[vocab.update({word:index}) for index, word in enumerate(vocab_list)]
encoded_train_data = []
def encode_word2int(data):
  word2int = []

  for text in data:
    tokens = str(text).lower().split()
# Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    word2int.append([vocab[word] for word in tokens])
  return word2int
encoded_train_data = encode_word2int(df.REVIEW)

import numpy as np
MAX_SEQ_LEN = int(np.median(review_lengths))
# MAX_SEQ_LEN = 100

X = []
for sentence in encoded_train_data:
  if len(sentence) > MAX_SEQ_LEN:
    X.append(sentence[:MAX_SEQ_LEN])
  else:
    X.append(sentence + [0]*(MAX_SEQ_LEN-len(sentence)))

x = torch.Tensor(np.array(X))
y=(df.RATING == "positive").astype(int).tolist()


In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_other, y_train, y_other = train_test_split(x, y, test_size=0.4, random_state=30)
x_val, x_test, y_val, y_test = train_test_split(x, y, test_size=0.5, random_state=30)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(0)
train_set = TensorDataset(torch.from_numpy(np.array(x_train)), torch.from_numpy(np.array(y_train)))
val_set = TensorDataset(torch.from_numpy(np.array(x_val)), torch.from_numpy(np.array(y_val)))
test_set = TensorDataset(torch.from_numpy(np.array(x_test)), torch.from_numpy(np.array(y_test)))
batch_size = 64

train_loader = DataLoader(train_set, batch_size, pin_memory=True, shuffle=True)


In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, patience):
        super(LSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, dropout=0.6, num_layers = num_layers)
        self.fc = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(0.3)
        self.sig = nn.Sigmoid()
    def forward(self, x, h):
        x = x.long()
        embeds = self.embedding(x)
        lstm_out, hidden = self.lstm(embeds, h)
        lstm_out = lstm_out[:, -1, :] # getting the last time step output
        lstm_out = self.dropout(lstm_out)
        out = self.fc(lstm_out)
        out = self.sig(out)
        return out

In [ ]:
vocab_size = len(vocab)
embedding_dim = 50
hidden_dim = 20
num_layers = 2

epochs = 100
lr = 0.0001
device="cuda"
model = LSTM(vocab_size, embedding_dim, hidden_dim, num_layers, 2)
model.to(device)
optimizer = torch.optim.Adam(params = model.parameters())

In [ ]:
import matplotlib.pyplot as plt

loss_by_epoch=[]
min_loss=99999
for epoch in range(epochs):
    current_epoch=epoch
    cum_loss=0.0
    for texts, labels in train_loader:
        texts = texts.to(device)
        labels = labels.to(device)
        bs = labels.shape[0]
        zero_init = torch.zeros(num_layers, bs, hidden_dim).cuda()
        h_c = tuple([zero_init, zero_init])
        preds = model(texts, h_c)
        loss = nn.BCELoss()(preds.squeeze(), labels.float())
        optimizer.zero_grad()
        loss.backward()
        cum_loss+=loss.cpu().detach().numpy()
        optimizer.step()
    loss_by_epoch.append(cum_loss)
    with torch.no_grad():
       zero_init = torch.zeros(num_layers, len(x_val), hidden_dim).to("cuda")
       h_c = tuple([zero_init, zero_init])
       preds_val=model(x_val.to("cuda"), h_c)
       loss_val = nn.BCELoss()(preds_val.squeeze(), torch.from_numpy(np.array(y_val)).float().cuda())
       if loss_val <= min_loss:
          min_loss = loss_val
       elif (loss_val/min_loss) > 1.1:
          break

plt.plot(range(1,current_epoch+1),loss_by_epoch[1:])
plt.xlabel("number of epochs")
plt.ylabel("loss")

## Evaluate on Validation set

In [ ]:
zero_init = torch.zeros(num_layers, len(x_val), hidden_dim).to("cuda")
h_c = tuple([zero_init, zero_init])
preds=(model(x_val.to("cuda"), h_c).squeeze()).tolist()
preds=[int(p > 0.5) for p in preds]

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
precision_recall_fscore_support(y_val,preds,average="macro")[2]

## Once you have a final model evaluate on Test set

In [ ]:
zero_init = torch.zeros(num_layers, len(x_test), hidden_dim).to("cuda")
h_c = tuple([zero_init, zero_init])
preds=(model(x_test.to("cuda"), h_c).squeeze()).tolist()
preds=[int(p > 0.5) for p in preds]

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
precision_recall_fscore_support(y_test,preds,average="macro")[2]

# Task: Run model as it is. Note F-score.
### Try to improve performance using Validation set.
### Use bootstrap to see if it is a significant improvement over original score.


# Solution follows:

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import random
import numpy as np
import math

def model_predict(model, x):
  zero_init = torch.zeros(num_layers, len(x), hidden_dim).to("cuda")
  h_c = tuple([zero_init, zero_init])
  preds=(model(x.to("cuda"), h_c).squeeze())
  preds=[int(p > 0.5) for p in preds]
  return(preds)

def draw_bootstrap_sample(data):
      n=len(data[0])
      indices = random.choices(range(n),k=n)
      return((torch.stack([data[0][ind] for ind in indices]),[data[1][ind] for ind in indices]))

def bootstrap_fscore_ci(data, model, num_samples):
    scores = []
    for i in range(num_samples):
        data_bs = draw_bootstrap_sample(data)
        score=precision_recall_fscore_support(data_bs[1],model_predict(model,data_bs[0]),average="macro")[2]
        scores.append(score)
    scores=np.sort(scores)
    return (np.mean(scores),scores[math.floor(len(scores)*0.025)],scores[math.floor(len(scores)*0.975)])

In [ ]:
bootstrap_fscore_ci((x_test,y_test),model,100)